# Lab 2 Phase 3: Regularized ResNet-SE SR

Phase 3 notebook for investigating the validation-PSNR plateau.

This version keeps the ResNet-SE backbone, but adds three experiment levers:
- fair train-side PSNR measured under `model.eval()` on a fixed train subset
- ImageNet-backed synthetic train/val data augmentation from `Data/ImageNet`
- stronger anti-overfitting measures through architecture regularization and data manipulation

**Architecture:** Stem -> N x SEResBlock (Conv-BN-PReLU + SE attention) -> Tail + global skip.

**New in Phase 3:** Dropout2d, stochastic depth, patch-based training, synthetic LR generation for ImageNet, cutout, and per-source diagnostics.

**Kernel:** Select `Python 3.9 (torch)` as the Jupyter kernel before running.

In [22]:
from collections import defaultdict
from pathlib import Path
from typing import Optional
import hashlib
import io
import json
import math
import os
import random
import time
import warnings

warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from PIL import Image, ImageFilter, ImageOps
from torchvision import transforms

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading

Phase 3 keeps the original paired SR data, then augments it with an ImageNet-backed
synthetic same-resolution restoration set.

- Original train data: `Data/HR_train*` and `Data/LR_train*`
- Original val data: `Data/val/HR_val` and `Data/val/LR_val`
- ImageNet train data: `Data/ImageNet/imagenet_train20a` driven by `imagenet_train20.txt`
- ImageNet val data: `Data/ImageNet/imagenet_val20` driven by `imagenet_val20.txt`

For ImageNet samples, the notebook creates synthetic LR inputs from HR images using
blur, downsample-upsample, and JPEG degradation. Validation uses deterministic
degradation so the metric stays reproducible.

In [23]:
NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
if (NOTEBOOK_DIR / "Data").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "Data").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(f"Could not locate Data directory from {NOTEBOOK_DIR}")

DATA_ROOT = PROJECT_ROOT / "Data"
IMAGE_ROOT = DATA_ROOT / "ImageNet"
OUTPUT_DIR = NOTEBOOK_DIR / "runs" / "resnet_se_sr_phase3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HR_TRAIN_ROOT = DATA_ROOT / "HR_train"
LR_TRAIN_ROOT = DATA_ROOT / "LR_train"
HR_VAL_DIR = DATA_ROOT / "val" / "HR_val"
LR_VAL_DIR = DATA_ROOT / "val" / "LR_val"
IMAGENET_TRAIN_ROOT = IMAGE_ROOT / "imagenet_train20a"
IMAGENET_VAL_ROOT = IMAGE_ROOT / "imagenet_val20"
IMAGENET_TRAIN_LIST = IMAGE_ROOT / "imagenet_train20.txt"
IMAGENET_VAL_LIST = IMAGE_ROOT / "imagenet_val20.txt"

for required in [
    HR_TRAIN_ROOT, LR_TRAIN_ROOT, HR_VAL_DIR, LR_VAL_DIR,
    IMAGENET_TRAIN_ROOT, IMAGENET_VAL_ROOT, IMAGENET_TRAIN_LIST, IMAGENET_VAL_LIST,
]:
    assert required.exists(), f"Missing required path: {required}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")


def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    hr_subdirs = sorted([d for d in hr_root.iterdir() if d.is_dir()])
    pairs = []
    for hr_sub in hr_subdirs:
        suffix = hr_sub.name.replace("HR_train", "")
        lr_sub = lr_root / f"LR_train{suffix}"
        if not lr_sub.exists():
            print(f"  WARNING: no matching LR subfolder for {hr_sub.name}")
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_sub.glob("*.png"))}
        lr_imgs = {p.stem: p for p in sorted(lr_sub.glob("*.png"))}
        for stem in sorted(set(hr_imgs) & set(lr_imgs)):
            pairs.append((lr_imgs[stem], hr_imgs[stem], f"{hr_sub.name}/{stem}"))
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    hr_imgs = {p.stem: p for p in sorted(hr_dir.glob("*.png"))}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.glob("*.png"))}
    common = sorted(set(hr_imgs) & set(lr_imgs))
    return [(lr_imgs[s], hr_imgs[s], s) for s in common]


def read_imagenet_manifest(path: Path) -> list[tuple[str, int]]:
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 2:
            continue
        rows.append((parts[0], int(parts[1])))
    return rows


def collect_imagenet_records(rows: list[tuple[str, int]], root: Path, split: str) -> list[dict]:
    records = []
    missing = 0
    for filename, class_id in rows:
        synset = filename.split("_")[0]
        path = (root / synset / filename) if split == "train" else (root / filename)
        if not path.exists():
            missing += 1
            continue
        records.append({
            "path": path,
            "stem": path.stem,
            "class_id": class_id,
            "split": split,
            "synset": synset,
        })
    if missing:
        print(f"WARNING: skipped {missing} missing ImageNet {split} files")
    return records


train_pairs = collect_paired_by_subfolder(LR_TRAIN_ROOT, HR_TRAIN_ROOT)
val_pairs = collect_paired_flat(LR_VAL_DIR, HR_VAL_DIR)
imagenet_train_rows = read_imagenet_manifest(IMAGENET_TRAIN_LIST)
imagenet_val_rows = read_imagenet_manifest(IMAGENET_VAL_LIST)
imagenet_train_records = collect_imagenet_records(imagenet_train_rows, IMAGENET_TRAIN_ROOT, split="train")
imagenet_val_records = collect_imagenet_records(imagenet_val_rows, IMAGENET_VAL_ROOT, split="val")

print(f"Original paired train samples: {len(train_pairs)}")
print(f"Original paired val samples:   {len(val_pairs)}")
print(f"ImageNet train samples:       {len(imagenet_train_records)}")
print(f"ImageNet val samples:         {len(imagenet_val_records)}")

Project root: /Users/cyrilgoud/Desktop/repos/personal/Lab 2
Output dir:   /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3
Original paired train samples: 3036
Original paired val samples:   100
ImageNet train samples:       6000
ImageNet val samples:         1000


In [24]:
DATA_CFG = {
    "train_patch_size": 224,
    "eval_size": 256,
    "random_scale_pad": 32,
    "cutout_prob": 0.35,
    "cutout_ratio": 0.18,
    "lr_noise_prob": 0.30,
    "lr_noise_std": 0.015,
    "lr_blur_prob": 0.70,
    "jpeg_prob": 0.50,
    "jpeg_quality_min": 40,
    "jpeg_quality_max": 90,
    "downsample_scales": (2, 3, 4),
    "imagenet_train_limit": 3000,
    "imagenet_val_limit": 300,
    "train_eval_subset_size": 128,
}
BATCH_SIZE = 8
NUM_WORKERS = 0
SEED = 255
TO_TENSOR = transforms.ToTensor()


def seeded_rng(key: str) -> random.Random:
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return random.Random(int(digest[:16], 16))


def take_manifest_subset(records: list[dict], limit: Optional[int], seed: int) -> list[dict]:
    if limit is None or limit >= len(records):
        return list(records)
    rng = random.Random(seed)
    items = list(records)
    rng.shuffle(items)
    return items[:limit]


def random_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random):
    if lr_img.width == size and lr_img.height == size:
        return lr_img, hr_img
    x0 = rng.randint(0, lr_img.width - size)
    y0 = rng.randint(0, lr_img.height - size)
    box = (x0, y0, x0 + size, y0 + size)
    return lr_img.crop(box), hr_img.crop(box)


def random_crop_single(img: Image.Image, size: int, rng: random.Random):
    if img.width == size and img.height == size:
        return img
    x0 = rng.randint(0, img.width - size)
    y0 = rng.randint(0, img.height - size)
    return img.crop((x0, y0, x0 + size, y0 + size))


def augment_pair(lr_img: Image.Image, hr_img: Image.Image, rng: random.Random):
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
        hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
        hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        rot = {1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k]
        lr_img = lr_img.transpose(rot)
        hr_img = hr_img.transpose(rot)
    return lr_img, hr_img


def augment_single(img: Image.Image, rng: random.Random):
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        img = img.transpose({1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k])
    return img


def jpeg_roundtrip(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def degrade_from_hr(hr_img: Image.Image, rng: random.Random, cfg: dict) -> Image.Image:
    lr_img = hr_img.copy()
    if rng.random() < cfg["lr_blur_prob"]:
        lr_img = lr_img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.2, 1.2)))
    scale = rng.choice(cfg["downsample_scales"])
    small = (max(1, hr_img.width // scale), max(1, hr_img.height // scale))
    lr_img = lr_img.resize(small, resample=BICUBIC).resize(hr_img.size, resample=BICUBIC)
    if rng.random() < cfg["jpeg_prob"]:
        lr_img = jpeg_roundtrip(lr_img, rng.randint(cfg["jpeg_quality_min"], cfg["jpeg_quality_max"]))
    return lr_img


def apply_tensor_regularization(lr_t: torch.Tensor, rng: random.Random, cfg: dict, train: bool) -> torch.Tensor:
    if not train:
        return lr_t
    if rng.random() < cfg["lr_noise_prob"]:
        lr_t = (lr_t + torch.randn_like(lr_t) * cfg["lr_noise_std"]).clamp(0.0, 1.0)
    if rng.random() < cfg["cutout_prob"]:
        _, h, w = lr_t.shape
        cut = max(8, int(min(h, w) * cfg["cutout_ratio"]))
        x0 = rng.randint(0, w - cut)
        y0 = rng.randint(0, h - cut)
        fill = lr_t.mean().item()
        lr_t[:, y0:y0 + cut, x0:x0 + cut] = fill
    return lr_t


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool, data_cfg: dict, source_name: str):
        self.pairs = pairs
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path, stem = self.pairs[idx]
        lr_img = Image.open(lr_path).convert("RGB")
        hr_img = Image.open(hr_path).convert("RGB")
        rng = random.Random(SEED + idx) if self.train else seeded_rng(f"{self.source_name}:{stem}")
        if self.train:
            lr_img, hr_img = random_crop_pair(lr_img, hr_img, self.data_cfg["train_patch_size"], rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            lr_img = ImageOps.fit(lr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, stem, self.source_name


class ImageNetSyntheticSRDataset(Dataset):
    def __init__(self, records: list[dict], train: bool, data_cfg: dict, source_name: str):
        self.records = records
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        hr_img = Image.open(record["path"]).convert("RGB")
        rng = random.Random(SEED + idx) if self.train else seeded_rng(f"{self.source_name}:{record['stem']}")
        if self.train:
            base_size = max(self.data_cfg["eval_size"], self.data_cfg["train_patch_size"] + self.data_cfg["random_scale_pad"])
            hr_img = ImageOps.fit(hr_img, (base_size, base_size), method=BICUBIC)
            hr_img = random_crop_single(hr_img, self.data_cfg["train_patch_size"], rng)
            hr_img = augment_single(hr_img, rng)
        else:
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        lr_img = degrade_from_hr(hr_img, rng, self.data_cfg)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, record["stem"], self.source_name


def make_fixed_subset_loader(dataset: Dataset, subset_size: int, batch_size: int, seed: int) -> DataLoader:
    subset_size = min(subset_size, len(dataset))
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    subset = Subset(dataset, indices[:subset_size])
    return DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)


imagenet_train_used = take_manifest_subset(imagenet_train_records, DATA_CFG["imagenet_train_limit"], SEED)
imagenet_val_used = take_manifest_subset(imagenet_val_records, DATA_CFG["imagenet_val_limit"], SEED)

paired_train_dataset = PairedSRDataset(train_pairs, train=True, data_cfg=DATA_CFG, source_name="paired_train")
paired_train_eval_dataset = PairedSRDataset(train_pairs, train=False, data_cfg=DATA_CFG, source_name="paired_train")
paired_val_dataset = PairedSRDataset(val_pairs, train=False, data_cfg=DATA_CFG, source_name="paired_val")
imagenet_train_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=True, data_cfg=DATA_CFG, source_name="imagenet_train")
imagenet_train_eval_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=False, data_cfg=DATA_CFG, source_name="imagenet_train")
imagenet_val_dataset = ImageNetSyntheticSRDataset(imagenet_val_used, train=False, data_cfg=DATA_CFG, source_name="imagenet_val")

train_dataset = ConcatDataset([paired_train_dataset, imagenet_train_dataset])
train_eval_dataset = ConcatDataset([paired_train_eval_dataset, imagenet_train_eval_dataset])
val_dataset = ConcatDataset([paired_val_dataset, imagenet_val_dataset])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
train_eval_loader = make_fixed_subset_loader(train_eval_dataset, DATA_CFG["train_eval_subset_size"], BATCH_SIZE, SEED)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
paired_val_loader = DataLoader(paired_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
imagenet_val_loader = DataLoader(imagenet_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

lr_batch, hr_batch, names, sources = next(iter(train_loader))
print(f"Combined train dataset: {len(train_dataset)} samples")
print(f"  paired train:         {len(paired_train_dataset)}")
print(f"  ImageNet train:       {len(imagenet_train_dataset)}")
print(f"Combined val dataset:   {len(val_dataset)} samples")
print(f"  paired val:           {len(paired_val_dataset)}")
print(f"  ImageNet val:         {len(imagenet_val_dataset)}")
print(f"Train-eval subset:      {len(train_eval_loader.dataset)} samples")
print(f"LR batch shape:         {tuple(lr_batch.shape)}")
print(f"HR batch shape:         {tuple(hr_batch.shape)}")
print(f"Sample names:           {list(names[:3])}")
print(f"Sample sources:         {list(sources[:3])}")

Combined train dataset: 6036 samples
  paired train:         3036
  ImageNet train:       3000
Combined val dataset:   400 samples
  paired val:           100
  ImageNet val:         300
Train-eval subset:      128 samples
LR batch shape:         (8, 3, 224, 224)
HR batch shape:         (8, 3, 224, 224)
Sample names:           ['n02415577_60041', 'n02423022_3237', 'n02117135_8490']
Sample sources:         ['imagenet_train', 'imagenet_train', 'imagenet_train']


## 2. Model Definition: Regularized ResNet-SE SR

Phase 3 keeps the SE residual design, but adds anti-overfitting capacity controls.

- modestly reduced width/depth versus the Phase 2 baseline
- `Dropout2d` inside the residual branch
- light stochastic depth on deeper blocks
- global residual learning: `output = LR + predicted_delta`

The training model contains regularization layers, while the export model later reuses
the same weights with dropout and drop-path disabled.

In [25]:
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob <= 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x * random_tensor / keep_prob


class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, mid, 1, bias=True)
        self.act = nn.PReLU(num_parameters=mid)
        self.fc2 = nn.Conv2d(mid, channels, 1, bias=True)
        self.gate = nn.Hardsigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.pool(x)
        w = self.act(self.fc1(w))
        w = self.gate(self.fc2(w))
        return x * w


class SEResBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4, dropout: float = 0.0, drop_path: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.act = nn.PReLU(num_parameters=channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0.0 else nn.Identity()
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.dropout(out)
        out = self.drop_path(out)
        return identity + out


class ResNetSESR(nn.Module):
    def __init__(self, num_blocks: int = 8, channels: int = 40, reduction: int = 4, dropout: float = 0.10, max_drop_path: float = 0.05):
        super().__init__()
        block_drop = torch.linspace(0.0, max_drop_path, steps=num_blocks).tolist() if num_blocks > 1 else [max_drop_path]
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(num_parameters=channels),
        )
        self.body = nn.Sequential(*[
            SEResBlock(channels, reduction=reduction, dropout=dropout, drop_path=block_drop[i])
            for i in range(num_blocks)
        ])
        self.tail = nn.Conv2d(channels, 3, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        feat = self.stem(x)
        feat = self.body(feat)
        delta = self.tail(feat)
        return identity + delta


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


MODEL_CFG = {
    "num_blocks": 8,
    "channels": 40,
    "reduction": 4,
    "dropout": 0.10,
    "max_drop_path": 0.05,
}

model = ResNetSESR(**MODEL_CFG).to(device)
with torch.no_grad():
    dummy = torch.randn(1, 3, DATA_CFG["train_patch_size"], DATA_CFG["train_patch_size"], device=device)
    out = model(dummy)

print(f"Model config:  {MODEL_CFG}")
print(f"Parameters:    {count_parameters(model):,}")
print(f"Input shape:   {tuple(dummy.shape)}")
print(f"Output shape:  {tuple(out.shape)}")

Model config:  {'num_blocks': 8, 'channels': 40, 'reduction': 4, 'dropout': 0.1, 'max_drop_path': 0.05}
Parameters:    241,163
Input shape:   (1, 3, 224, 224)
Output shape:  (1, 3, 224, 224)


## 3. Training Infrastructure

Phase 3 now uses explicit `L1` supervision, fair eval-mode train PSNR logging, and
periodic checkpoint archival.

It logs two train-side signals:
- `train_psnr_online`: the same fast in-epoch train metric used during optimization
- `train_eval_psnr`: a fairer metric computed under `model.eval()` on a fixed train subset

The gap between those two values helps separate BatchNorm and train-mode inflation from
true generalization problems, while `last.pt`, `best.pt`, and periodic `epoch_XXX.pt`
snapshots keep rollback points available during long runs.

In [26]:
TRAIN_CFG = {
    "epochs": 40,
    "batch_size": BATCH_SIZE,
    "lr": 3e-4,
    "weight_decay": 2e-4,
    "warmup_epochs": 3,
    "min_lr_ratio": 0.10,
    "checkpoint_interval": 10,
    "seed": SEED,
    "early_stop_patience": 8,
}
RESUME_TRAINING = True
LOSS_FN = nn.L1Loss()

print(f"Config:  {TRAIN_CFG}")
print("Loss:    L1")
print(f"Resume:  {RESUME_TRAINING}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


def lr_for_epoch(epoch: int, total: int, base_lr: float, warmup: int, min_ratio: float) -> float:
    if warmup > 0 and epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total - warmup - 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_ratio
    return min_lr + (base_lr - min_lr) * cosine


def train_one_epoch(model, loader, optimizer, cfg):
    model.train()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(lr_img)
        loss = LOSS_FN(pred, hr_img)
        loss.backward()
        optimizer.step()
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        with torch.no_grad():
            total_psnr += compute_psnr(pred.detach(), hr_img).sum().item()
        n += bs
    return {
        "train_loss": total_loss / max(1, n),
        "train_psnr_online": total_psnr / max(1, n),
    }


@torch.no_grad()
def evaluate_loader(model, loader, cfg, split_name: str):
    model.eval()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        pred = model(lr_img)
        loss = LOSS_FN(pred, hr_img)
        psnr = compute_psnr(pred, hr_img)
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        total_psnr += psnr.sum().item()
        n += bs
    return {
        f"{split_name}_loss": total_loss / max(1, n),
        f"{split_name}_psnr": total_psnr / max(1, n),
    }


def save_checkpoint(path, model, optimizer, epoch, metrics, best_psnr, best_epoch):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "best_psnr": best_psnr,
        "best_epoch": best_epoch,
        "model_cfg": MODEL_CFG,
        "train_cfg": TRAIN_CFG,
        "data_cfg": DATA_CFG,
        "loss_name": "l1",
    }, path)


def load_history(metrics_path: Path) -> list[dict]:
    if not metrics_path.exists():
        return []
    return [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]


def resolve_best_epoch(history: list[dict], checkpoint: dict, best_ckpt_path: Path, best_psnr: float) -> int:
    if history:
        return int(max(history, key=lambda row: row["val_psnr"])["epoch"])

    saved_best_epoch = int(checkpoint.get("best_epoch", -1))
    if saved_best_epoch >= 0:
        return saved_best_epoch

    if best_ckpt_path.exists():
        best_ckpt = torch.load(best_ckpt_path, map_location="cpu")
        if isinstance(best_ckpt, dict):
            if int(best_ckpt.get("best_epoch", -1)) >= 0:
                return int(best_ckpt["best_epoch"])
            if int(best_ckpt.get("epoch", -1)) >= 0:
                return int(best_ckpt["epoch"])

    metrics = checkpoint.get("metrics", {})
    metric_psnr = metrics.get("val_psnr")
    if metric_psnr is not None and math.isclose(float(metric_psnr), float(best_psnr), rel_tol=0.0, abs_tol=1e-12):
        return int(metrics.get("epoch", -1))

    return -1


def should_archive_checkpoint(epoch: int, total_epochs: int, interval: int, should_stop: bool) -> bool:
    return (epoch % interval == 0) or (epoch == total_epochs) or should_stop


def fit(model, train_loader, train_eval_loader, val_loader, output_dir, cfg=TRAIN_CFG, resume=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    last_ckpt_path = output_dir / "last.pt"
    best_ckpt_path = output_dir / "best.pt"

    optimizer = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    start_epoch = 0
    best_psnr = float("-inf")
    best_epoch = -1
    history = []
    cumulative_time = 0.0

    if resume and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        best_psnr = ckpt.get("best_psnr", float("-inf"))
        history = load_history(metrics_path)
        best_epoch = resolve_best_epoch(history, ckpt, best_ckpt_path, best_psnr)
        cumulative_time = sum(row.get("seconds", 0.0) for row in history)
        print(f"RESUMING from epoch {start_epoch}/{cfg['epochs']}")
        print(f"Best checkpoint remains at epoch {best_epoch} with val_psnr={best_psnr:.3f} dB")
    else:
        set_seed(cfg["seed"])
        metrics_path.write_text("")
        print("FRESH START")

    print(f"\n{'epoch':>5} | {'lr':>8} | {'train_loss':>10} {'train_online':>12} {'train_eval':>11} | {'val_psnr':>8} | {'best':>8} | {'time':>5}")
    print("-" * 98)

    for epoch in range(start_epoch, cfg["epochs"]):
        epoch_lr = lr_for_epoch(epoch, cfg["epochs"], cfg["lr"], cfg["warmup_epochs"], cfg["min_lr_ratio"])
        for group in optimizer.param_groups:
            group["lr"] = epoch_lr

        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, optimizer, cfg)
        train_eval_metrics = evaluate_loader(model, train_eval_loader, cfg, split_name="train_eval")
        val_metrics = evaluate_loader(model, val_loader, cfg, split_name="val")
        elapsed = time.time() - t0
        cumulative_time += elapsed

        row = {
            "epoch": epoch + 1,
            "lr": epoch_lr,
            "train_loss": train_metrics["train_loss"],
            "train_psnr_online": train_metrics["train_psnr_online"],
            "train_eval_psnr": train_eval_metrics["train_eval_psnr"],
            "val_loss": val_metrics["val_loss"],
            "val_psnr": val_metrics["val_psnr"],
            "seconds": round(elapsed, 1),
        }
        history.append(row)
        with metrics_path.open("a") as f:
            f.write(json.dumps(row) + "\n")

        is_best = row["val_psnr"] > best_psnr
        if is_best:
            best_psnr = row["val_psnr"]
            best_epoch = epoch + 1

        stale_epochs = (epoch + 1) - best_epoch
        should_stop = stale_epochs >= cfg["early_stop_patience"]

        save_checkpoint(last_ckpt_path, model, optimizer, epoch + 1, row, best_psnr, best_epoch)
        if is_best:
            save_checkpoint(best_ckpt_path, model, optimizer, epoch + 1, row, best_psnr, best_epoch)
        if should_archive_checkpoint(epoch + 1, cfg["epochs"], cfg["checkpoint_interval"], should_stop):
            archive_path = output_dir / f"epoch_{epoch + 1:03d}.pt"
            save_checkpoint(archive_path, model, optimizer, epoch + 1, row, best_psnr, best_epoch)

        marker = " *" if is_best else ""
        print(
            f"{epoch+1:3d}/{cfg['epochs']:3d} | {epoch_lr:.2e} | "
            f"{row['train_loss']:10.6f} {row['train_psnr_online']:10.3f} {row['train_eval_psnr']:10.3f} | "
            f"{row['val_psnr']:8.3f} | {best_psnr:8.3f} | {elapsed:5.1f}s{marker}"
        )

        if should_stop:
            print(f"Early stopping after {cfg['early_stop_patience']} epochs without validation improvement.")
            break

    best_row = max(history, key=lambda r: r["val_psnr"])
    print(f"\nTraining complete. Best epoch {best_row['epoch']} with val_psnr={best_row['val_psnr']:.3f} dB")
    print(f"Total training time: {cumulative_time/60:.1f} min")
    print(f"Checkpoints: {output_dir}")
    return history

Config:  {'epochs': 40, 'batch_size': 8, 'lr': 0.0003, 'weight_decay': 0.0002, 'warmup_epochs': 3, 'min_lr_ratio': 0.1, 'checkpoint_interval': 10, 'seed': 255, 'early_stop_patience': 8}
Loss:    L1
Resume:  True


## 4. Smoke Test

Quick run on small subsets to confirm the mixed-data pipeline, regularized model,
and fair train-eval logging all work before committing to a long training run.

In [27]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_cfg = {**TRAIN_CFG, "epochs": 2, "early_stop_patience": 2}
    smoke_train = DataLoader(Subset(train_dataset, list(range(min(32, len(train_dataset))))), batch_size=4, shuffle=True, num_workers=0)
    smoke_train_eval = DataLoader(Subset(train_eval_dataset, list(range(min(16, len(train_eval_dataset))))), batch_size=4, shuffle=False, num_workers=0)
    smoke_val = DataLoader(Subset(val_dataset, list(range(min(16, len(val_dataset))))), batch_size=4, shuffle=False, num_workers=0)

    smoke_model = ResNetSESR(**MODEL_CFG).to(device)
    smoke_opt = AdamW(smoke_model.parameters(), lr=smoke_cfg["lr"], weight_decay=smoke_cfg["weight_decay"])

    for ep in range(smoke_cfg["epochs"]):
        tm = train_one_epoch(smoke_model, smoke_train, smoke_opt, smoke_cfg)
        te = evaluate_loader(smoke_model, smoke_train_eval, smoke_cfg, split_name="train_eval")
        vm = evaluate_loader(smoke_model, smoke_val, smoke_cfg, split_name="val")
        print(
            f"smoke epoch {ep+1}: train_loss={tm['train_loss']:.6f} | "
            f"train_online={tm['train_psnr_online']:.3f} dB | "
            f"train_eval={te['train_eval_psnr']:.3f} dB | "
            f"val={vm['val_psnr']:.3f} dB"
        )

    del smoke_model, smoke_opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Smoke test passed.")

smoke epoch 1: train_loss=0.480363 | train_online=9.078 dB | train_eval=19.219 dB | val=16.399 dB
smoke epoch 2: train_loss=0.336594 | train_online=10.902 dB | train_eval=20.358 dB | val=17.696 dB
Smoke test passed.


## 5. Full Training

The default Phase 3 setup starts fresh and writes to a separate run directory.

The most important metric line to watch is:
- `train_psnr_online` vs `train_eval_psnr`: estimates how much train-mode logging is inflating the apparent train score
- `train_eval_psnr` vs `val_psnr`: estimates the real generalization gap after removing that artifact

In [28]:
model = ResNetSESR(**MODEL_CFG).to(device)
print(f"Target: {TRAIN_CFG['epochs']} epochs")
print(f"Train samples: {len(train_dataset)} | Train-eval subset: {len(train_eval_loader.dataset)} | Val samples: {len(val_dataset)}")
print(f"Resume: {RESUME_TRAINING} | Output: {OUTPUT_DIR}\n")

history = fit(model, train_loader, train_eval_loader, val_loader, OUTPUT_DIR, cfg=TRAIN_CFG, resume=RESUME_TRAINING)

Target: 40 epochs
Train samples: 6036 | Train-eval subset: 128 | Val samples: 400
Resume: True | Output: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3

RESUMING from epoch 38/40
Best checkpoint remains at epoch 38 with val_psnr=25.510 dB

epoch |       lr | train_loss train_online  train_eval | val_psnr |     best |  time
--------------------------------------------------------------------------------------------------
 39/ 40 | 3.05e-05 |   0.037031     25.938     27.120 |   25.523 |   25.523 | 811.0s *
 40/ 40 | 3.00e-05 |   0.037018     25.943     27.152 |   25.543 |   25.543 | 829.1s *

Training complete. Best epoch 40 with val_psnr=25.543 dB
Total training time: 1004.2 min
Checkpoints: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3


## 6. Diagnostics: Per-Source PSNR And Difficulty

This section helps answer the core plateau question.

- `train_eval` vs `val`: how much of the gap survives under `eval()`
- `paired_val` vs `imagenet_val`: whether one validation source is materially harder
- `input_psnr`: how difficult the LR-to-HR restoration problem is before the model even runs

In [29]:
def load_checkpoint(model, checkpoint_path, map_location="cpu"):
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    return ckpt


@torch.no_grad()
def collect_psnr_records(model, loader, max_items: Optional[int] = None) -> list[dict]:
    model.eval()
    records = []
    for lr_img, hr_img, names, sources in loader:
        lr_img = lr_img.to(device)
        hr_img = hr_img.to(device)
        pred = model(lr_img)
        pred_psnr = compute_psnr(pred, hr_img).cpu().tolist()
        input_psnr = compute_psnr(lr_img, hr_img).cpu().tolist()
        target_std = hr_img.std(dim=(1, 2, 3)).cpu().tolist()
        for name, source, p_pred, p_in, std in zip(names, sources, pred_psnr, input_psnr, target_std):
            records.append({
                "name": name,
                "source": source,
                "pred_psnr": float(p_pred),
                "input_psnr": float(p_in),
                "target_std": float(std),
            })
            if max_items is not None and len(records) >= max_items:
                return records
    return records


def summarize_records(title: str, records: list[dict]) -> None:
    if not records:
        print(f"{title}: no records")
        return
    pred = torch.tensor([r["pred_psnr"] for r in records], dtype=torch.float32)
    baseline = torch.tensor([r["input_psnr"] for r in records], dtype=torch.float32)
    contrast = torch.tensor([r["target_std"] for r in records], dtype=torch.float32)
    print(
        f"{title}: n={len(records)} | model_psnr={pred.mean().item():.3f} dB | "
        f"baseline_input_psnr={baseline.mean().item():.3f} dB | hr_std={contrast.mean().item():.4f}"
    )
    print(
        f"  quantiles: p10={torch.quantile(pred, 0.10).item():.3f} | "
        f"p50={torch.quantile(pred, 0.50).item():.3f} | p90={torch.quantile(pred, 0.90).item():.3f}"
    )
    hardest = sorted(records, key=lambda r: r["pred_psnr"])[:5]
    print("  hardest samples:")
    for row in hardest:
        print(f"    {row['source']}: {row['name']} -> {row['pred_psnr']:.3f} dB")


best_ckpt = OUTPUT_DIR / "best.pt"
if best_ckpt.exists():
    diag_model = ResNetSESR(**MODEL_CFG).to(device)
    ckpt = load_checkpoint(diag_model, best_ckpt, map_location=device)
    print(f"Loaded best checkpoint from epoch {ckpt.get('epoch', '?')}")

    train_eval_records = collect_psnr_records(diag_model, train_eval_loader, max_items=DATA_CFG["train_eval_subset_size"])
    paired_val_records = collect_psnr_records(diag_model, paired_val_loader)
    imagenet_val_records = collect_psnr_records(diag_model, imagenet_val_loader)
    combined_val_records = collect_psnr_records(diag_model, val_loader)

    summarize_records("train_eval", train_eval_records)
    summarize_records("paired_val", paired_val_records)
    summarize_records("imagenet_val", imagenet_val_records)
    summarize_records("combined_val", combined_val_records)
else:
    print(f"Run training first so {best_ckpt} exists.")

Loaded best checkpoint from epoch 40
train_eval: n=128 | model_psnr=27.152 dB | baseline_input_psnr=26.581 dB | hr_std=0.2249
  quantiles: p10=21.203 | p50=27.095 | p90=33.838
  hardest samples:
    paired_train: HR_train3/000712 -> 18.207 dB
    imagenet_train: n02013706_21431 -> 18.349 dB
    paired_train: HR_train2/000169 -> 18.545 dB
    paired_train: HR_train4/000387 -> 19.167 dB
    imagenet_train: n02676566_9407 -> 19.289 dB
paired_val: n=100 | model_psnr=21.696 dB | baseline_input_psnr=21.336 dB | hr_std=0.2432
  quantiles: p10=18.120 | p50=21.989 | p90=24.434
  hardest samples:
    paired_val: 000557 -> 16.634 dB
    paired_val: 000186 -> 16.692 dB
    paired_val: 000156 -> 17.741 dB
    paired_val: 000120 -> 17.748 dB
    paired_val: 000166 -> 17.777 dB
imagenet_val: n=300 | model_psnr=26.825 dB | baseline_input_psnr=25.893 dB | hr_std=0.2265
  quantiles: p10=21.583 | p50=26.540 | p90=32.291
  hardest samples:
    imagenet_val: n02013706_4575 -> 18.188 dB
    imagenet_val: n0

## 7. Decision Guide

Use this summary after diagnostics to decide what to prioritize next.

- Large `train_psnr_online - train_eval_psnr`: metric inflation from train-mode logging is a major factor
- Large `train_eval_psnr - val_psnr`: real overfitting remains after removing the logging artifact
- Much lower `imagenet_val` than `paired_val`: validation difficulty or source shift is a major factor
- Flat combined validation plus high train-eval PSNR: prioritize stronger regularization and/or stronger train degradations

In [30]:
def recommend_priority(history: list[dict], train_eval_records: list[dict], paired_val_records: list[dict], imagenet_val_records: list[dict]) -> None:
    if not history:
        print("Run training first to generate recommendations.")
        return

    last = history[-1]
    train_mode_gap = last["train_psnr_online"] - last["train_eval_psnr"]
    generalization_gap = last["train_eval_psnr"] - last["val_psnr"]
    paired_mean = torch.tensor([r["pred_psnr"] for r in paired_val_records], dtype=torch.float32).mean().item() if paired_val_records else float("nan")
    imagenet_mean = torch.tensor([r["pred_psnr"] for r in imagenet_val_records], dtype=torch.float32).mean().item() if imagenet_val_records else float("nan")

    print(f"train-mode inflation gap: {train_mode_gap:.3f} dB")
    print(f"real train-eval vs val gap: {generalization_gap:.3f} dB")
    print(f"paired_val mean: {paired_mean:.3f} dB")
    print(f"imagenet_val mean: {imagenet_mean:.3f} dB")

    if train_mode_gap > 1.0:
        print("Priority 1: keep using fair eval-mode train PSNR for analysis; the old train metric is materially inflated.")
    if generalization_gap > 1.5:
        print("Priority 2: regularization and stronger train-time degradation still matter after correcting the metric artifact.")
    if paired_val_records and imagenet_val_records and (paired_mean - imagenet_mean) > 0.75:
        print("Priority 3: ImageNet-backed validation is harder; treat source shift and degradation realism as first-class issues.")
    if generalization_gap <= 1.5 and abs(paired_mean - imagenet_mean) <= 0.75:
        print("Priority 4: architecture may be close to its ceiling; consider loss tuning or data scaling before larger structural changes.")


if best_ckpt.exists():
    recommend_priority(history if "history" in globals() else [], train_eval_records, paired_val_records, imagenet_val_records)

train-mode inflation gap: -1.209 dB
real train-eval vs val gap: 1.609 dB
paired_val mean: 21.696 dB
imagenet_val mean: 26.825 dB
Priority 2: regularization and stronger train-time degradation still matter after correcting the metric artifact.


## 7. ONNX Export

For export, the notebook rebuilds the same residual architecture with dropout and stochastic depth disabled.
That keeps the inference graph clean while preserving the learned convolution and attention weights,
including the global residual path `output = LR + predicted_delta` used during training.

In [31]:
def export_to_onnx(checkpoint_path, onnx_path, verify=False, sample_loader=None):
    checkpoint_path = Path(checkpoint_path)
    onnx_path = Path(onnx_path)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)

    export_cfg = {**MODEL_CFG, "dropout": 0.0, "max_drop_path": 0.0}
    export_model = ResNetSESR(**export_cfg).to(device)
    ckpt = load_checkpoint(export_model, checkpoint_path, map_location=device)
    export_model.eval()

    if "metrics" in ckpt:
        print(
            f"Checkpoint epoch {ckpt['metrics'].get('epoch', '?')}, "
            f"val_psnr={ckpt['metrics'].get('val_psnr', '?'):.3f} dB"
        )
    print("Residual path preserved for export: output = LR + predicted_delta")

    dummy = torch.randn(1, 3, DATA_CFG["eval_size"], DATA_CFG["eval_size"], device=device)
    export_kwargs = dict(
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
    )

    try:
        torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(export_model, dummy, str(onnx_path), **export_kwargs)

    print(f"Exported to {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")

    if onnx is not None:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        print("ONNX checker: passed")
    else:
        print("onnx package not installed, skipped checker")

    if verify and ort is not None and sample_loader is not None:
        sample_lr, _, _, _ = next(iter(sample_loader))
        sample_lr = sample_lr[:1].to(device)
        with torch.no_grad():
            torch_out = export_model(sample_lr).cpu().numpy()
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = sess.run(["output"], {"input": sample_lr.cpu().numpy()})[0]
        diff = abs(torch_out - ort_out)
        print(f"Parity check: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")

    return onnx_path


best_ckpt = OUTPUT_DIR / "best.pt"
if best_ckpt.exists():
    onnx_path = export_to_onnx(best_ckpt, OUTPUT_DIR / "best.onnx", verify=True, sample_loader=paired_val_loader)
    print(f"Ready for Mobilint compilation: {onnx_path}")
else:
    print(f"Run training first so {best_ckpt} exists.")

Checkpoint epoch 40, val_psnr=25.543 dB
Residual path preserved for export: output = LR + predicted_delta
Exported to /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3/best.onnx (956 KB)
ONNX checker: passed
Parity check: max_diff=0.00000036, mean_diff=0.00000004
Ready for Mobilint compilation: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3/best.onnx


## 8. Calibration Set Export

Quantization quality depends heavily on calibration coverage, so this section builds a
deterministic LR calibration subset from the training data instead of sampling randomly.

The selector scores candidate LR inputs by:
- mean brightness
- contrast
- simple texture energy from spatial gradients
- source coverage across paired and ImageNet-backed training data

It then exports a manifest plus PNG tensors next to the ONNX artifact for downstream
Mobilint or other NPU calibration flows.

In [32]:
CALIBRATION_CFG = {
    "num_samples": 128,
    "seed": SEED,
    "output_subdir": "calibration",
}


def slugify_name(text: str) -> str:
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in text)
    cleaned = "_".join(part for part in cleaned.split("_") if part)
    return cleaned[:80] or "sample"


@torch.no_grad()
def score_lr_tensor(lr_t: torch.Tensor) -> dict:
    gray = lr_t.float().mean(dim=0)
    brightness = float(gray.mean().item())
    contrast = float(gray.std().item())
    grad_x = gray[:, 1:] - gray[:, :-1]
    grad_y = gray[1:, :] - gray[:-1, :]
    texture = float(0.5 * (grad_x.abs().mean().item() + grad_y.abs().mean().item()))
    return {
        "brightness": brightness,
        "contrast": contrast,
        "texture": texture,
    }


@torch.no_grad()
def collect_calibration_candidates(dataset_map: dict[str, Dataset]) -> list[dict]:
    records = []
    for dataset_key, dataset in dataset_map.items():
        for dataset_index in range(len(dataset)):
            lr_t, _, name, source = dataset[dataset_index]
            stats = score_lr_tensor(lr_t)
            records.append({
                "dataset_key": dataset_key,
                "dataset_index": dataset_index,
                "name": name,
                "source": source,
                **stats,
            })
    return records


def assign_tertile_bins(records: list[dict], metric_name: str) -> None:
    values = torch.tensor([row[metric_name] for row in records], dtype=torch.float32)
    low_cut = torch.quantile(values, 1.0 / 3.0).item()
    high_cut = torch.quantile(values, 2.0 / 3.0).item()
    for row in records:
        value = row[metric_name]
        if value <= low_cut:
            row[f"{metric_name}_bin"] = "low"
        elif value >= high_cut:
            row[f"{metric_name}_bin"] = "high"
        else:
            row[f"{metric_name}_bin"] = "mid"


def select_diverse_calibration_subset(records: list[dict], num_samples: int, seed: int) -> list[dict]:
    if not records:
        return []

    tagged = [dict(row) for row in records]
    assign_tertile_bins(tagged, "brightness")
    assign_tertile_bins(tagged, "texture")

    rng = random.Random(seed)
    buckets = defaultdict(list)
    for row in tagged:
        bucket_key = (row["source"], row["brightness_bin"], row["texture_bin"])
        buckets[bucket_key].append(row)

    ordered_keys = sorted(buckets)
    for key in ordered_keys:
        rng.shuffle(buckets[key])

    target = min(num_samples, len(tagged))
    selected = []
    seen = set()
    while len(selected) < target:
        made_progress = False
        for key in ordered_keys:
            bucket = buckets[key]
            while bucket:
                row = bucket.pop()
                row_id = (row["dataset_key"], row["dataset_index"])
                if row_id in seen:
                    continue
                seen.add(row_id)
                selected.append(row)
                made_progress = True
                break
            if len(selected) >= target:
                break
        if not made_progress:
            break
    return selected


def summarize_calibration_subset(records: list[dict]) -> dict:
    source_counts = defaultdict(int)
    brightness_bins = defaultdict(int)
    texture_bins = defaultdict(int)
    for row in records:
        source_counts[row["source"]] += 1
        brightness_bins[row["brightness_bin"]] += 1
        texture_bins[row["texture_bin"]] += 1
    return {
        "count": len(records),
        "source_counts": dict(sorted(source_counts.items())),
        "brightness_bins": dict(sorted(brightness_bins.items())),
        "texture_bins": dict(sorted(texture_bins.items())),
        "brightness_range": [min(row["brightness"] for row in records), max(row["brightness"] for row in records)],
        "contrast_range": [min(row["contrast"] for row in records), max(row["contrast"] for row in records)],
        "texture_range": [min(row["texture"] for row in records), max(row["texture"] for row in records)],
    }


def export_calibration_artifacts(records: list[dict], dataset_map: dict[str, Dataset], output_dir: Path, cfg: dict):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calibration_dir = output_dir / cfg["output_subdir"]
    images_dir = calibration_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    to_pil = transforms.ToPILImage()
    manifest = []
    batch = []
    for idx, row in enumerate(records):
        dataset = dataset_map[row["dataset_key"]]
        lr_t, _, name, source = dataset[row["dataset_index"]]
        rel_path = Path("images") / f"{idx:03d}_{slugify_name(source)}_{slugify_name(name)}.png"
        image_path = calibration_dir / rel_path
        to_pil(lr_t).save(image_path)
        batch.append(lr_t)
        manifest.append({
            **row,
            "image_path": str(rel_path),
        })

    torch.save(torch.stack(batch), calibration_dir / "calibration_inputs.pt")
    summary = summarize_calibration_subset(records)
    (calibration_dir / "manifest.json").write_text(json.dumps({
        "config": cfg,
        "summary": summary,
        "samples": manifest,
    }, indent=2))

    print(f"Calibration samples exported: {len(records)}")
    print(f"Calibration dir: {calibration_dir}")
    print(f"Source coverage: {summary['source_counts']}")
    print(f"Brightness bins: {summary['brightness_bins']}")
    print(f"Texture bins: {summary['texture_bins']}")
    return calibration_dir, manifest, summary


calibration_datasets = {
    "paired_train": paired_train_eval_dataset,
    "imagenet_train": imagenet_train_eval_dataset,
}
calibration_candidates = collect_calibration_candidates(calibration_datasets)
selected_calibration = select_diverse_calibration_subset(
    calibration_candidates,
    num_samples=CALIBRATION_CFG["num_samples"],
    seed=CALIBRATION_CFG["seed"],
)
calibration_dir, calibration_manifest, calibration_summary = export_calibration_artifacts(
    selected_calibration,
    calibration_datasets,
    OUTPUT_DIR,
    CALIBRATION_CFG,
)
print(f"Calibration manifest: {calibration_dir / 'manifest.json'}")
print(f"Calibration tensor batch: {calibration_dir / 'calibration_inputs.pt'}")

Calibration samples exported: 128
Calibration dir: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3/calibration
Source coverage: {'imagenet_train': 65, 'paired_train': 63}
Brightness bins: {'high': 44, 'low': 42, 'mid': 42}
Texture bins: {'high': 43, 'low': 43, 'mid': 42}
Calibration manifest: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3/calibration/manifest.json
Calibration tensor batch: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 3/runs/resnet_se_sr_phase3/calibration/calibration_inputs.pt
